In [1]:
# Bootstrap the import path so `src/` can be imported from notebooks.
from pathlib import Path
import sys

# If you're running the notebook from lumbar-coordinate/notebooks/, go one level up.
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent.resolve()
else:
    # If you open the notebook from the repo root, use that directly.
    PROJECT_ROOT = Path.cwd().resolve()

# Add the project root (which contains `src/`) to sys.path once per session.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\hyeon\Documents\miniconda_medimg_env\lumbar-coordinate


In [2]:
# --- 03_validate_report.ipynb ---
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.config import Paths, TrainConfig
from src.utils import set_seed
from src.transforms import build_transforms
from src.dataset import LumbarDataset
from src.model import get_model
from src.metrics import mae_per_level
from src.visualization import overlay_points

# Choose the run to analyze
PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == 'notebooks') else Path.cwd()
paths = Paths.build(project_root=PROJECT_ROOT)
RUN_DIR = max((paths.runs_dir).glob("run_*"))  # pick latest; or set explicitly

# Load splits
processed_dir = PROJECT_ROOT / "data" / "processed"
df_val = pd.read_csv(processed_dir / "val.csv")

# Dataset/Dataloader
cfg = TrainConfig()
_, val_tfms = build_transforms(cfg.image_size)
val_ds = LumbarDataset(df_val, paths.data_images_dir, list(cfg.levels), transform=val_tfms, allow_missing=True)
val_dl = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

# Load model
model = get_model(backbone=cfg.backbone, pretrained=False, out_dim=cfg.out_dim)
ckpt = torch.load(RUN_DIR / "checkpoints" / "best.pt", map_location="cpu")
model.load_state_dict(ckpt["model"])
model.eval()

# Metrics + qualitative overlays
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

all_pred, all_tgt, all_fn = [], [], []
with torch.no_grad():
    for x, y, fns in val_dl:
        x = x.to(device).float()
        out = model(x).cpu().numpy()
        all_pred.append(out)
        all_tgt.append(y.numpy())
        all_fn.extend(list(fns))

pred = np.vstack(all_pred)   # (N, 10)
tgt  = np.vstack(all_tgt)    # (N, 10)

# Per-level MAE (normalized)
per_level = np.mean([mae_per_level(torch.tensor(pred[i:i+1]), torch.tensor(tgt[i:i+1]))
                     for i in range(pred.shape[0])], axis=0)
report_df = pd.DataFrame({"level": cfg.levels, "mae_norm": per_level})
display(report_df)

# Save table & overlays
fig_dir = RUN_DIR / "figures"
fig_dir.mkdir(exist_ok=True, parents=True)
report_df.to_csv(RUN_DIR / "val_per_level_mae.csv", index=False)

# Save a few overlays
# (use the dataset's internally stored image paths by re-building a small map)
name_to_path = {fn: p for fn, p, _ in val_ds.samples}
for i in range(min(24, len(all_fn))):
    fn = all_fn[i]
    img_path = name_to_path[fn]
    overlay_points(img_path, pred[i], tgt[i], cfg.levels, save_path=fig_dir / f"{Path(fn).stem}_overlay.jpg")

print("Saved:", RUN_DIR / "val_per_level_mae.csv", "| Overlays in:", fig_dir)

C:\Users\hyeon\AppData\Local\Temp\ipykernel_53236\1515895728.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(RUN_DIR / "checkpoints" / "best.pt", map_

,level,mae_norm
0,L1/L2,0.015175
1,L2/L3,0.012585
2,L3/L4,0.010439
3,L4/L5,0.009554
4,L5/S1,0.009363


Saved: C:\Users\hyeon\Documents\miniconda_medimg_env\lumbar-coordinate\runs\run_20260202_1546\val_per_level_mae.csv | Overlays in: C:\Users\hyeon\Documents\miniconda_medimg_env\lumbar-coordinate\runs\run_20260202_1546\figures


In [3]:
# After you compute pred, tgt, and all_fn in validation:
import numpy as np, pandas as pd
levels = ["L1/L2","L2/L3","L3/L4","L4/L5","L5/S1"]

rows = []
for i, fn in enumerate(all_fn):
    row = {"filename": fn}
    pr = pred[i].reshape(5,2); gtv = tgt[i].reshape(5,2)
    dists = []
    for j,lvl in enumerate(levels):
        row[f"pred_x_{lvl}"], row[f"pred_y_{lvl}"] = float(pr[j,0]), float(pr[j,1])
        row[f"gt_x_{lvl}"],   row[f"gt_y_{lvl}"]   = float(gtv[j,0]), float(gtv[j,1])
        dists.append(np.sqrt(((pr[j,0]-gtv[j,0])**2) + ((pr[j,1]-gtv[j,1])**2)))
    row["mean_error"] = float(np.mean(dists))
    rows.append(row)

pd.DataFrame(rows).to_csv(RUN_DIR / "predictions.csv", index=False)
print("Saved:", RUN_DIR/"predictions.csv")

Saved: C:\Users\hyeon\Documents\miniconda_medimg_env\lumbar-coordinate\runs\run_20260202_1546\predictions.csv


In [4]:
# --- PDF report generation cell ---
from pathlib import Path
import json
from src.config import Paths
from src.report_pdf import generate_pdf_report

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == 'notebooks') else Path.cwd()
paths = Paths.build(project_root=PROJECT_ROOT)

# Choose run directory (latest or explicit)
RUN_DIR = max((paths.runs_dir).glob("run_*"))  # or set explicitly

# Methods dictionary: pull from config.json to auto-fill
with open(RUN_DIR / "config.json","r") as f:
    cfg_dict = json.load(f)

methods = {
    "Backbone": cfg_dict.get("backbone","resnet18"),
    "Pretrained": cfg_dict.get("pretrained", True),
    "Image size": cfg_dict.get("image_size", 320),
    "Batch size": cfg_dict.get("batch_size", 32),
    "Optimizer": "AdamW",
    "Base loss": f"SmoothL1 (beta={cfg_dict.get('huber_beta', 0.01)})",
    "Scheduler": cfg_dict.get("scheduler","plateau"),
    "LR": cfg_dict.get("lr", 1e-3),
    "Weight decay": cfg_dict.get("weight_decay", 1e-4),
    "Early stopping": f"patience={cfg_dict.get('early_stop_patience',5)}, min_delta={cfg_dict.get('early_stop_min_delta',1e-4)}",
    "Ordering prior": f"lambda={cfg_dict.get('order_reg_lambda',0.1)}, margin={cfg_dict.get('order_reg_margin',0.0)}"
}

# Overlay images (take first 6)
fig_dir = RUN_DIR / "figures"
overlay_paths = sorted(fig_dir.glob("*.jpg"))[:6]

# Splits and metrics
processed_dir = PROJECT_ROOT / "data" / "processed"
train_csv = processed_dir / "train.csv"
val_csv = processed_dir / "val.csv"
metrics_csv = RUN_DIR / "metrics.csv"

out_pdf = paths.reports_dir / f"report_{RUN_DIR.name}.pdf"

generate_pdf_report(
    project_root=PROJECT_ROOT,
    run_dir=RUN_DIR,
    train_csv=train_csv,
    val_csv=val_csv,
    metrics_csv=metrics_csv,
    methods_dict=methods,
    overlay_paths=overlay_paths,
    out_pdf=out_pdf
)

print("PDF saved to:", out_pdf)

PDF saved to: C:\Users\hyeon\Documents\miniconda_medimg_env\lumbar-coordinate\reports\report_run_20260202_1546.pdf
